# 00 · Data Ingestion (2015–2025 Raw Datasets)

**Project:** AgriShield — Early Warning System for Crop Yield & Livestock Forage Risk in Kenya  
**Notebook Goal:** Load and inspect our **10 raw datasets (2015–2025)** from `data_pipeline/data/` before cleaning them in the next notebook.

### Data Sources (Proposal Table 3.1)
1. **Geospatial & County Codes:** `kenya_counties.geojson` and `kenya_county_crosswalk.csv` (all 47 Kenyan counties).
2. **Forage & Vegetation (NASA MODIS):** `ken_ndvi_subnational_2015_2025_raw.csv` (pasture greenness / NDVI).
3. **Weather & Soil Moisture (CHIRPS & ERA5-Land):** Rainfall, temperature, and root-zone soil moisture (`2015–2025`).
4. **Crop Production & Demographics (KNBS & FAOSTAT):** Historical crop harvests (`2015–2025`) and county farming households (`2019 Census`).

---
### Step 1: Import Libraries & Check Our 10 Raw Files
In the cell below, we import `pandas`, `numpy`, `json`, and `Path`, point to our `../data` folder, and check that all 10 raw files are present.

In [4]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Point to the folder where our 10 raw datasets are stored
DATA_DIR = Path("../data")

# List all files in the data folder and check their sizes
files = sorted(DATA_DIR.glob("*"))

file_summary = pd.DataFrame([
    {
        "File Name": f.name,
        "Size (KB)": round(f.stat().st_size / 1024, 1)
    }
    for f in files if not f.name.startswith(".")
])

print(f"Total raw files found: {len(file_summary)}")
display(file_summary)

Total raw files found: 11


,File Name,Size (KB)
0,era5land_precipitation_2015_2025_raw.csv,705.5
1,era5land_soil_moisture_runoff_2015_2025_raw.csv,1073.8
2,faostat_kenya_crop_production_2015_2025_raw.csv,3.7
3,ken_chirps_rainfall_subnational_2015_2025_raw.csv,3952.4
4,ken_ndvi_subnational_2015_2025_raw.csv,2032.0
5,kenya_counties.geojson,6823.0
6,kenya_county_crosswalk.csv,1.4
7,knbs_2019_census_agriculture_households_raw.csv,3.4
8,knbs_agricultural_production_2015_2025_raw.csv,4.4
9,modis_era5_climate_vegetation_monthly_2015_202...,1827.9


### Step 2: Load Geospatial Boundaries & the 47-County Crosswalk

Why do we need these two files?
1. **`kenya_counties.geojson`**: Contains the map boundaries (polygons) for all **47 counties in Kenya**.
2. **`kenya_county_crosswalk.csv`**: Links different county IDs (`knbs_code`, `pcode` like `KE001`–`KE047`, and uppercase names) to the standard names of all **47 Kenyan counties** so we can merge all datasets accurately.

In [10]:
# 1. Load the 47-county crosswalk CSV

crosswalk_df = pd.read_csv(DATA_DIR / "kenya_county_crosswalk.csv")
print(f"Crosswalk shape: {crosswalk_df.shape} (Total counties: {crosswalk_df['county'].nunique()})")
display(crosswalk_df.head(10))

# 2. Load the Kenya Counties GeoJSON and verify all 47 counties are present

with open(DATA_DIR / "kenya_counties.geojson", "r") as f:
    kenya_geojson = json.load(f)

geojson_counties = sorted([feat["properties"]["county"] for feat in kenya_geojson["features"]])
print(f"\nTotal county polygons in GeoJSON: {len(geojson_counties)}")
print("First 10 counties in GeoJSON:", geojson_counties[:10])

Crosswalk shape: (47, 5) (Total counties: 47)


,county,knbs_code,pcode,desinventar_code,desinventar_name
0,Mombasa,1,KE001,28,MOMBASA
1,Kwale,2,KE002,19,KWALE
2,Kilifi,3,KE003,14,KILIFI
3,Tana River,4,KE004,40,TANA-RIVER
4,Lamu,5,KE005,21,LAMU
5,Taita-Taveta,6,KE006,39,TAITA-TAVETA
6,Garissa,7,KE007,6,GARISSA
7,Wajir,8,KE008,46,WAJIR
8,Mandera,9,KE009,24,MANDERA
9,Marsabit,10,KE010,25,MARSABIT



Total county polygons in GeoJSON: 47
First 10 counties in GeoJSON: ['Baringo', 'Bomet', 'Bungoma', 'Busia', 'Elgeyo-Marakwet', 'Embu', 'Garissa', 'Homa Bay', 'Isiolo', 'Kajiado']


### Step 3: Load Raw NASA MODIS NDVI & CHIRPS Rainfall Data (2015–2025)

Next, we load our two main satellite time-series datasets from `2015` to `2025`:
1. **`ken_ndvi_subnational_2015_2025_raw.csv`**: Raw 10-day (dekadal) **Normalized Difference Vegetation Index (NDVI)** from NASA MODIS (`vim` = vegetation greenness, `viq` = vegetation anomaly %). This measures pasture and crop health across all 47 counties.
2. **`ken_chirps_rainfall_subnational_2015_2025_raw.csv`**: Raw 10-day (dekadal) **CHIRPS v2.0 Rainfall** (`rfh` = 10-day rainfall in mm, `r1h` = 1-month rolling rainfall, `r3h` = 3-month rolling rainfall, `rfq` = rainfall anomaly %).

Notice that both files are raw and uncleaned: they contain a metadata tag row at the top, use `PCODE`s instead of county names, and include both county (`adm_level = 1`) and sub-county (`adm_level = 2`) records. We inspect these raw issues here so we know what to clean in `01_data_cleaning.ipynb`.

In [ ]:
# 1. Load raw NASA MODIS NDVI (2015-2025

ndvi_raw = pd.read_csv(DATA_DIR / "ken_ndvi_subnational_2015_2025_raw.csv", low_memory=False)
print("====== Raw NASA MODIS NDVI (2015-2025) ======")
print(f"Shape: {ndvi_raw.shape}")
print(f"Columns: ", list(ndvi_raw.columns))
display(ndvi_raw.head(10))

# 2. Load raw CHIRPS Rainfall (2015-2025)

chirps_raw = pd.read_csv(DATA_DIR / "ken_chirps_rainfall_subnational_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw CHIRPS Subnational Rainfall (2015-2025) ======")
print(f"Shape: {chirps_raw.shape}")
print(f"Columns: ", list(chirps_raw.columns))
display(chirps_raw.head(10))
d


====== Raw NASA MODIS NDVI (2015-2025) ======
Shape: (32076, 8)
Columns:  ['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'vim', 'vim_avg', 'viq']


,date,adm_level,adm_id,PCODE,n_pixels,vim,vim_avg,viq
0,2015-01-01,1,51325,KE019,427.0,0.675912,0.692547,97.759790
1,2015-01-11,1,51325,KE019,427.0,0.647016,0.672289,96.500970
2,2015-01-21,1,51325,KE019,427.0,0.618583,0.651686,95.282360
3,2015-02-01,1,51325,KE019,427.0,0.592075,0.630918,94.295570
4,2015-02-11,1,51325,KE019,427.0,0.572094,0.613749,93.724290
5,2015-02-21,1,51325,KE019,427.0,0.560143,0.602420,93.519950
6,2015-03-01,1,51325,KE019,427.0,0.554891,0.597363,93.439255
7,2015-03-11,1,51325,KE019,427.0,0.558923,0.600382,93.625410
8,2015-03-21,1,51325,KE019,427.0,0.577016,0.613856,94.450640
9,2015-04-01,1,51325,KE019,427.0,0.607171,0.636140,95.778030



====== Raw CHIRPS Subnational Rainfall (2015-2025) ======
Shape: (32076, 15)
Columns:  ['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg', 'r1h', 'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version']


,date,adm_level,adm_id,PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,2015-01-01,1,51325,KE019,427.0,5.583138,15.759407,50.241220,66.389694,299.259950,346.43607,50.979960,77.379820,86.576190,final
1,2015-01-11,1,51325,KE019,427.0,5.615925,19.294770,39.461357,62.209213,262.824370,328.69415,43.696340,66.153660,80.260430,final
2,2015-01-21,1,51325,KE019,427.0,6.721312,16.265417,17.920374,51.319595,234.887600,292.37277,55.119120,40.696980,80.668980,final
3,2015-02-01,1,51325,KE019,427.0,3.690866,12.719282,16.028103,48.279470,185.487140,245.99680,49.047510,39.467552,75.892260,final
4,2015-02-11,1,51325,KE019,427.0,29.962530,18.768618,40.374710,47.753320,152.065580,206.99610,147.095350,86.012990,74.088900,final
5,2015-02-21,1,51325,KE019,427.0,2.939110,11.827791,36.592506,43.315693,129.194400,171.64575,47.178562,86.084880,75.968090,final
6,2015-03-01,1,51325,KE019,427.0,15.056206,29.220140,47.957848,59.816550,114.227165,174.48572,58.609367,81.704210,66.427110,final
7,2015-03-11,1,51325,KE019,427.0,11.004684,28.629663,29.000000,69.677600,108.836070,179.64012,47.590977,45.529050,61.652940,final
8,2015-03-21,1,51325,KE019,427.0,27.456675,41.624825,53.517563,99.474630,108.030440,194.10991,69.612434,56.011265,56.767864,final
9,2015-04-01,1,51325,KE019,427.0,94.208430,73.952380,132.669780,144.206860,196.655730,252.30289,125.656040,92.267730,78.372900,final


### Step 4: Load Raw ERA5-Land Soil Moisture, Precipitation & MODIS Temperature Data (2015–2025)

In this step, we load the three meteorological and land-surface datasets that provide **temperature** and **root-zone soil moisture** across all **47 counties** from `2015` to `2025`:
1. **`era5land_soil_moisture_runoff_2015_2025_raw.csv`**: Raw Google Earth Engine (`ECMWF/ERA5-Land`) monthly county data containing `volumetric_soil_water_layer_1` (`0–7 cm` top-soil moisture) and `volumetric_soil_water_layer_2` (`7–28 cm` root-zone moisture).
2. **`era5land_precipitation_2015_2025_raw.csv`**: Raw `ERA5-Land` monthly precipitation (`mean`) across all 47 counties.
3. **`modis_era5_climate_vegetation_monthly_2015_2025_raw.csv`**: Raw monthly grid observations containing `max_temp_c`, `min_temp_c`, `mean_temp_c`, `humidity_rh_pct`, `soil_moisture_m3m3`, `rain_mm`, and `ndvi` (`2015–2025`).

**Raw Data Issues to Note for Cleaning Later:**
- The Earth Engine files store county names in uppercase under the column `FIRST_COUN` and include unnecessary GEE system columns (`system:index`, `COUNT`, `.geo`).
- The monthly grid file contains multiple `(lat, lon)` points per county and some border points labeled `"Unknown"`.

In [19]:
# 1. Load raw ERA5-Land Soil Moisture & Runoff (2015-2025)

era5_soil_raw = pd.read_csv(DATA_DIR / "era5land_soil_moisture_runoff_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw ERA5-Lnd Soil Moisture & Runoff (2015-2025) ======")
print(f"Shape: {era5_soil_raw.shape} | Unique Counties: {era5_soil_raw['FIRST_COUN'].nunique()}")
display(era5_soil_raw.head(10))

# 2. Load raw ERA5-Land Monthly Precipitation (2015-2025)

era5_precip_raw = pd.read_csv(DATA_DIR / "era5land_precipitation_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw ERA5-Lnd Monthly Precipitation (2015-2025) ======")
print(f"Shape: {era5_precip_raw.shape} | Unique Counties: {era5_precip_raw['FIRST_COUN'].nunique()}")
display(era5_precip_raw.head(10))

# 3. Load raw ERA5-Land Monthly Temperature (2015-2025)

climate_grid_raw = pd.read_csv(DATA_DIR / "modis_era5_climate_vegetation_monthly_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw MODIS-ERA5 Climate & Vegetation (2015-2025) ======")
print(f"Shape: {climte_grid_raw.shape} | Data Range: {climate_grid_raw['month'].min()} to {climate_grid_raw['month'].max()}")
print("Columns: ", list(climate_grid_raw.columns))
display(climate_grid_raw.head(10))


====== Raw ERA5-Lnd Soil Moisture & Runoff (2015-2025) ======
Shape: (6204, 11) | Unique Counties: 47


,system:index,COUNT,COUNTYCO,FIRST_COUN,month,runoff_sum,surface_runoff_sum,volumetric_soil_water_layer_1,volumetric_soil_water_layer_2,year,.geo
0,201501_00000000000000000001,1,2,BOMET,1,0.001547,1.240025e-04,0.316361,0.356298,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,201501_00000000000000000007,1,8,ISIOLO,1,0.000005,4.077893e-06,0.118284,0.146438,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,201501_00000000000000000008,1,9,KAJIADO,1,0.003257,7.992425e-04,0.210551,0.247311,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,201501_0000000000000000000e,1,15,KIRINYAGA,1,0.047341,1.093390e-02,0.298412,0.319060,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,201501_00000000000000000012,1,19,KWALE,1,0.000040,2.765110e-05,0.138207,0.163966,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
5,201501_00000000000000000013,1,20,LAIKIPIA,1,0.004864,2.090232e-04,0.202922,0.248741,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
6,201501_00000000000000000014,1,21,LAMU,1,0.000015,9.474877e-06,0.131368,0.166329,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
7,201501_00000000000000000015,1,22,MACHAKOS,1,0.000386,1.236886e-04,0.221251,0.248706,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
8,201501_00000000000000000016,1,23,MAKUENI,1,0.000400,1.400956e-04,0.209727,0.241682,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
9,201501_00000000000000000017,1,24,MANDERA,1,0.000001,5.174853e-07,0.125035,0.208156,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"



====== Raw ERA5-Lnd Monthly Precipitation (2015-2025) ======
Shape: (6204, 8) | Unique Counties: 47


,system:index,COUNT,COUNTYCO,FIRST_COUN,mean,month,year,.geo
0,201501_00000000000000000001,1,2,BOMET,0.014601,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,201501_00000000000000000007,1,8,ISIOLO,0.001334,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,201501_00000000000000000008,1,9,KAJIADO,0.034009,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,201501_0000000000000000000e,1,15,KIRINYAGA,0.083745,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,201501_00000000000000000012,1,19,KWALE,0.008694,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
5,201501_00000000000000000013,1,20,LAIKIPIA,0.015942,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
6,201501_00000000000000000014,1,21,LAMU,0.003371,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
7,201501_00000000000000000015,1,22,MACHAKOS,0.020040,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
8,201501_00000000000000000016,1,23,MAKUENI,0.016437,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"
9,201501_00000000000000000017,1,24,MANDERA,0.000134,1,2015,"{""type"":""MultiPoint"",""coordinates"":[]}"



====== Raw MODIS-ERA5 Climate & Vegetation (2015-2025) ======
Shape: (10817, 22) | Data Range: 2015-01 to 2025-12
Columns:  ['month', 'county', 'lat', 'lon', 'max_temp_c', 'min_temp_c', 'mean_temp_c', 'dtr_c', 'elevation_m', 'ndvi', 'evi', 'veg_cover_pct', 'forest_pct', 'savanna_pct', 'wetland_pct', 'cropland_pct', 'urban_pct', 'surface_water_occurrence_pct', 'rain_mm', 'humidity_rh_pct', 'soil_moisture_m3m3', 'wind_speed_ms']


,month,county,lat,lon,max_temp_c,min_temp_c,mean_temp_c,dtr_c,elevation_m,ndvi,...,forest_pct,savanna_pct,wetland_pct,cropland_pct,urban_pct,surface_water_occurrence_pct,rain_mm,humidity_rh_pct,soil_moisture_m3m3,wind_speed_ms
0,2015-01,Unknown,5.008108,35.326249,48.61,20.97,33.3200,27.64,456.0,0.1884,...,0.0,0.0,0.0000,0.0,0.0,0.0,5.2842,24.4881,0.0367,0.8839
1,2015-01,Turkana,5.008108,35.371164,47.75,23.59,34.0225,24.16,461.0,0.2090,...,0.0,0.0,0.0000,0.0,0.0,0.0,5.1694,24.5167,0.0320,1.0813
2,2015-01,Unknown,4.963192,35.146585,43.23,22.59,31.4850,20.64,459.0,0.2633,...,0.0,0.0,0.0000,0.0,0.0,1.0,5.7358,24.1499,0.0836,0.5725
3,2015-01,Marsabit,4.424203,36.134732,26.87,26.09,26.3450,0.78,361.0,-0.0933,...,0.0,0.0,0.0000,0.0,0.0,99.0,5.6737,46.7393,0.1013,3.0748
4,2015-01,Marsabit,4.424203,36.179648,27.13,25.39,26.1750,1.74,361.0,0.0632,...,0.0,0.0,1.4917,0.0,0.0,98.0,6.1153,36.7474,0.1020,2.7305
5,2015-01,Wajir,3.660635,39.772909,37.33,22.73,30.0525,14.60,953.0,0.3438,...,0.0,0.0,0.0000,0.0,0.0,0.0,5.0424,26.7778,0.0812,3.4160
6,2015-01,Wajir,3.615719,39.727993,41.05,20.07,30.9675,20.98,869.0,0.2892,...,0.0,0.0,0.0000,0.0,0.0,0.0,5.1475,27.1054,0.0813,3.6323
7,2015-01,West Pokot,2.627572,35.146585,42.49,22.23,32.1675,20.26,778.0,0.2115,...,0.0,0.0,0.0000,0.0,0.0,0.0,0.0000,25.6164,0.0225,1.2281
8,2015-01,West Pokot,2.582656,35.146585,41.75,22.89,31.8950,18.86,833.0,0.2375,...,0.0,0.0,0.0000,0.0,0.0,0.0,0.0000,25.6164,0.0225,1.2281
9,2015-01,Samburu,2.492825,36.718637,36.79,26.97,31.6450,9.82,633.0,0.1547,...,0.0,0.0,0.0000,0.0,0.0,72.0,6.5082,36.6847,0.0220,2.3312


### Step 5: Load Raw Crop Production (FAOSTAT & KNBS) & County Demographic Data (2015–2025)

Next, we load the three socioeconomic and agricultural production datasets from **Table 3.1** of our Project Proposal:
1. **`faostat_kenya_crop_production_2015_2025_raw.csv`**: Annual crop yield (`kg/ha`), production (`tonnes`), and area harvested (`ha`) from **FAOSTAT (`2015–2025`)**.
2. **`knbs_agricultural_production_2015_2025_raw.csv`**: Marketed and physical agricultural & livestock output from the **KNBS Statistical Abstract & Economic Survey Chapter 7 (`2015–2025*`)**.
3. **`knbs_2019_census_agriculture_households_raw.csv`**: County-level distribution of households practicing farming, crop production, livestock rearing, aquaculture, fishing, and irrigation from the **KNBS 2019 Census (Volume IV, Table 2.19)**.

**Raw Data Issues to Note for Cleaning Later:**
- The FAOSTAT file includes neighboring East African countries (`Uganda`, `Ethiopia`, `Tanzania`) alongside `Kenya`, plus flag columns (`A`, `E`).
- The KNBS production and census tables store numbers as comma-formatted text strings (e.g., `"8,506.0"`, `"12,043,016"`), have provisional column headers (`2024*`, `2025*`), and include a national `"KENYA"` total row (`County_Code = 000`) and slash/hyphen county names (`TAITA/TAVETA`, `ELGEYO/MARAKWET`).

In [21]:
# 1. Load raw FAOSTAT Crop Production, Yield & Area Harvested (2015-2025)

faostat_raw = pd.read_csv(DATA_DIR / "faostat_kenya_crop_production_2015_2025_raw.csv")
print("=== 1. Raw FAOSTAT Crop Production & Yield (2015–2025) ===")
print(f"Shape: {faostat_raw.shape} | Countries in file: {faostat_raw['Country'].unique().tolist()}")
display(faostat_raw.head(5))

# 2. Load raw KNBS Chapter 7 Agricultural & Livestock Production (2015-2025*)

knbs_prod_raw = pd.read_csv(DATA_DIR / "knbs_agricultural_production_2015_2025_raw.csv")
print("\n=== 2. Raw KNBS Agricultural & Livestock Production (2015–2025*) ===")
print(f"Shape: {knbs_prod_raw.shape}")
display(knbs_prod_raw.head(6))

# 3. Load raw KNBS 2019 Census Agricultural & Livestock Households by County

census_demo_raw = pd.read_csv(DATA_DIR / "knbs_2019_census_agriculture_households_raw.csv")
print("\n=== 3. Raw KNBS 2019 Census Agricultural Households ===")
print(f"Shape: {census_demo_raw.shape} (Includes 1 national 'KENYA' total row + 47 counties)")
display(census_demo_raw.head(6))

=== 1. Raw FAOSTAT Crop Production & Yield (2015–2025) ===
Shape: (52, 12) | Countries in file: ['United Republic of Tanzania', 'Ethiopia', 'Kenya', 'Uganda']


,faostat,m49 code,Country,Item Code,Item,Year,Yield (kg/ha),Yield (kg/ha) flag,Production (tonnes),Production (tonnes) flag,Area harvested (ha),Area harvested (ha) flag
0,215,834,United Republic of Tanzania,56,Maize (corn),2015,1558.4,A,5902776.00,A,3787751,A
1,238,231,Ethiopia,56,Maize (corn),2015,2776.5,A,8211424.38,A,2957510,A
2,114,404,Kenya,56,Maize (corn),2015,1823.0,A,3825000.00,A,2098240,A
3,226,800,Uganda,56,Maize (corn),2015,2500.0,A,2812919.00,A,1125168,A
4,238,231,Ethiopia,56,Maize (corn),2016,2946.8,A,8846787.38,A,3002159,A



=== 2. Raw KNBS Agricultural & Livestock Production (2015–2025*) ===
Shape: (26, 15)


,Table_Ref,Sub_Sector,Commodity,Unit,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024*,2025*
0,Table 7.2,Cereals (Marketed Value),Maize,KSh Million,"8,506.0","7,891.5","6,326.8","9,987.1","10,663.4","6,996.7","7,694.4","10,127.0","13,828.2","12,410.5","14,120.8"
1,Table 7.2,Cereals (Marketed Value),Wheat,KSh Million,"4,192.4","4,561.2","3,418.9","6,124.5","6,892.0","6,659.9","5,773.9","9,879.0","11,245.6","10,890.2","11,980.4"
2,Table 7.2,Cereals (Marketed Value),Rice Paddy,KSh Million,"1,890.2","2,145.6","2,012.4","2,456.8","2,678.5","3,814.3","3,918.2","4,215.6","5,412.0","6,120.4","6,845.0"
3,Table 7.2,Cereals (Marketed Value),Barley and Other Cereals,KSh Million,"3,578.6","3,244.9","2,069.6","2,930.7","2,506.8","1,905.0","1,705.7","2,198.0","2,704.6","2,705.0","3,332.2"
4,Table 7.2,Cereals (Marketed Value),TOTAL CEREALS,KSh Million,"18,167.2","17,843.2","13,827.7","21,499.1","22,740.7","19,375.9","19,092.2","26,419.6","33,190.4","32,126.1","36,278.4"
5,Table 7.2,Temporary Industrial Crops,Sugar-cane,KSh Million,"22,398.1","23,940.6","20,145.8","21,890.4","23,412.6","29,254.0","31,420.5","38,912.4","44,120.8","46,890.2","51,240.6"



=== 3. Raw KNBS 2019 Census Agricultural Households ===
Shape: (48, 9) (Includes 1 national 'KENYA' total row + 47 counties)


,County_Code,County / Sub-County,Total Households,Farming Households,Crop Production,Livestock Production,Aquaculture,Fishing,Irrigation
0,0,KENYA,"12,043,016","6,354,211","5,556,416","4,722,135","29,325","106,695","369,679"
1,1,MOMBASA,"378,428","13,726","9,840","6,450",145,"2,450","1,120"
2,2,KWALE,"173,176","118,450","109,840","78,920",210,"6,890","3,450"
3,3,KILIFI,"298,472","169,135","156,890","108,450",340,"9,850","5,420"
4,4,TANA RIVER,"68,242","41,250","26,890","31,450",95,"2,890","6,890"
5,5,LAMU,"37,963","21,450","17,890","13,450",85,"4,890","1,120"


### Step 6: Raw Data Quality & Audit Summary (What We Need to Clean Next)

Before moving to **`01_data_cleaning.ipynb`**, we generate a single **Raw Data Audit Summary Table** checking the rows, columns, missing values, and specific cleaning tasks required for each dataset.

In [22]:
# Build a simple, clear audit summary of all tabular raw datasets
raw_datasets = {
    "County Crosswalk": (crosswalk_df, "Reference lookup for 47 counties (PCODE & KNBS codes)"),
    "MODIS NDVI (2015–2025)": (ndvi_raw, "Filter adm_level==1, map PCODE to county, aggregate dekads to monthly mean"),
    "CHIRPS Rainfall (2015–2025)": (chirps_raw, "Filter adm_level==1, map PCODE to county, sum dekadal rainfall per month"),
    "ERA5 Soil Moisture (2015–2025)": (era5_soil_raw, "Map uppercase FIRST_COUN to standard county, drop GEE system/.geo columns"),
    "ERA5 Precipitation (2015–2025)": (era5_precip_raw, "Map uppercase FIRST_COUN to standard county, drop GEE system/.geo columns"),
    "MODIS/ERA5 Climate Grid (2015–2025)": (climate_grid_raw, "Drop 'Unknown' border points, standardize county names, average per county-month"),
    "FAOSTAT Crops (2015–2025)": (faostat_raw, "Filter Country=='Kenya', keep clean numeric Yield & Production columns"),
    "KNBS Production (2015–2025*)": (knbs_prod_raw, "Strip commas from numbers, rename '2024*'/'2025*' to '2024'/'2025', reshape wide/long"),
    "KNBS 2019 Census Households": (census_demo_raw, "Drop 'KENYA' total row (000), strip commas, standardize 47 county names"),
}

audit_rows = []
for name, (df, cleaning_task) in raw_datasets.items():
    audit_rows.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Action Required in 01_data_cleaning.ipynb": cleaning_task,
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
print("\nNotebook 00 (Data Ingestion) Complete! Ready for 01_data_cleaning.ipynb.")

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Action Required in 01_data_cleaning.ipynb
0,County Crosswalk,47,5,0,0,Reference lookup for 47 counties (PCODE & KNBS...
1,MODIS NDVI (2015–2025),32076,8,0,0,"Filter adm_level==1, map PCODE to county, aggr..."
2,CHIRPS Rainfall (2015–2025),32076,15,0,0,"Filter adm_level==1, map PCODE to county, sum ..."
3,ERA5 Soil Moisture (2015–2025),6204,11,0,0,"Map uppercase FIRST_COUN to standard county, d..."
4,ERA5 Precipitation (2015–2025),6204,8,0,0,"Map uppercase FIRST_COUN to standard county, d..."
5,MODIS/ERA5 Climate Grid (2015–2025),10817,22,0,0,"Drop 'Unknown' border points, standardize coun..."
6,FAOSTAT Crops (2015–2025),52,12,0,0,"Filter Country=='Kenya', keep clean numeric Yi..."
7,KNBS Production (2015–2025*),26,15,0,0,"Strip commas from numbers, rename '2024*'/'202..."
8,KNBS 2019 Census Households,48,9,0,0,"Drop 'KENYA' total row (000), strip commas, st..."



Notebook 00 (Data Ingestion) Complete! Ready for 01_data_cleaning.ipynb.
